In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point


In [3]:
station_shp = r"C:\Users\Keitaro Ninomiya\Box\Research Notes (keitaro2@illinois.edu)\RailwayUnions\Processed_Data\5. 1861 England, Wales and Scotland rail stations\1861EngWalesScotRail_Stations.shp"

accident_csv = r"C:\Users\Keitaro Ninomiya\Box\Research Notes (keitaro2@illinois.edu)\RailwayUnions\Processed_Data\detailed_accidents_data.csv"

branch_csv = r"C:\Users\Keitaro Ninomiya\Box\Research Notes (keitaro2@illinois.edu)\RailwayUnions\Processed_Data\ASRS\BalanceSheets\1875\Results\georeferenced_railway_results.csv"


In [4]:
stations = gpd.read_file(station_shp)

# Ensure projected CRS (meters); British National Grid
stations = stations.to_crs(epsg=27700)


In [9]:
print(stations.columns)


Index(['Id', 'geometry'], dtype='object')


In [10]:
pip install geopy tqdm


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Keitaro Ninomiya\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [11]:
import pandas as pd

acc = pd.read_csv(accident_csv)

acc["location_clean"] = (
    acc["location"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z ]", "", regex=True)
    .str.strip()
)


In [13]:
UK_KEYWORDS = [
    "england", "scotland", "wales", "london", "york",
    "junction", "station", "near", "at", "between"
]

def looks_uk(x):
    x = x.lower()
    return any(k in x for k in UK_KEYWORDS)

acc["location_clean"] = (
    acc["location"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z ]", "", regex=True)
    .str.strip()
)

acc["likely_uk"] = acc["location_clean"].apply(looks_uk)

locations = acc.loc[acc["likely_uk"], "location_clean"].unique()

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import time

geolocator = Nominatim(user_agent="railway_accident_geocoder")
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1,
    max_retries=1,
    swallow_exceptions=True
)

geo_lookup = pd.DataFrame(geo_rows)

geo_lookup["geocode_success"] = geo_lookup["latitude"].notna()


In [14]:
geo_lookup.to_csv(
    "accident_location_geocoding.csv",
    index=False
)


In [15]:
acc = acc.merge(
    geo_lookup,
    on="location_clean",
    how="left"
)


In [ ]:
import geopandas as gpd

acc_gdf = gpd.GeoDataFrame(
    acc.dropna(subset=["longitude", "latitude"]),
    geometry=gpd.points_from_xy(acc.longitude, acc.latitude),
    crs="EPSG:4326"
).to_crs(epsg=27700)


In [16]:
print("Total accidents:", len(acc))
print("Likely UK:", acc["likely_uk"].sum())
print("Geocoded:", acc["latitude"].notna().sum())

acc.loc[acc["latitude"].isna(), "location"].value_counts().head(20)


Total accidents: 9212
Likely UK: 1902
Geocoded: 0


location
London Bridge            47
Preston                  47
Crewe                    41
Waterloo                 35
Carlisle                 33
Unknown location         32
Manchester Victoria      31
Victoria                 28
Kings Cross              28
Cannon Street            27
Wigan                    26
Birmingham New Street    25
Liverpool Street         25
York                     24
Leicester                23
Hatfield                 22
Accrington               22
Bolton                   21
Stafford                 21
Newcastle Central        21
Name: count, dtype: int64

In [17]:
stations = gpd.read_file(station_shp)
print(stations.columns)


Index(['Id', 'geometry'], dtype='object')


In [19]:
STATION_NAME_COL = "Station"   # CHANGE to the correct column
stations = stations.to_crs(epsg=27700)


In [20]:
stations["station_clean"] = (
    stations[STATION_NAME_COL]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z ]", "", regex=True)
    .str.strip()
)


KeyError: 'Station'